# Catastrophic Forgetting과 W&B 실전

> 파인튜닝의 대가 — 도메인 능력을 얻으면서 범용 능력을 잃는 **Catastrophic Forgetting**을 시연하고, 이를 **Replay Buffer**로 방지한다

Phase 4에서 SQL 파인튜닝된 모델이 범용 질문에서 얼마나 열화되는지 직접 확인하고,  
W&B(Weights & Biases) 실전 연동으로 학습 추적 수준을 높인다.

In [ ]:
# === 환경 설치 ===
!pip install datasets transformers peft accelerate trl bitsandbytes matplotlib wandb

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# GPU 확인
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("GPU 없음 - CPU 모드")

---
## 1. Catastrophic Forgetting이란?

모델을 특정 도메인(SQL)에 파인튜닝하면, 그 도메인의 능력은 올라가지만  
**원래 알고 있던 범용 능력이 퇴화**한다. 이것이 Catastrophic Forgetting이다.

| 상태 | SQL 능력 | 범용 능력 |
|------|---------|----------|
| 원본 모델 | 보통 | 높음 |
| SQL 파인튜닝 후 | **높음** | **하락** |
| Replay Buffer 적용 | 높음 | 유지 |

In [ ]:
# === 모델 로드 ===

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"모델: {model_name} (QLoRA 4-bit)")

In [ ]:
# === 추론 헬퍼 ===

def generate_response(model, tokenizer, prompt, max_new_tokens=150):
    """ChatML 형식으로 추론"""
    formatted = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(formatted, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# 범용 테스트 질문 (SQL과 무관)
general_prompts = [
    "Explain the difference between a list and a tuple in Python.",
    "What is photosynthesis?",
    "Write a short poem about the ocean.",
    "What are the main causes of climate change?",
]

# SQL 테스트 질문
sql_prompts = [
    "Write a SQL query to find the top 5 customers by total order amount.",
    "How do you join two tables in SQL?",
]

---
## 2. 원본 모델 Baseline

파인튜닝 전 원본 모델의 범용 능력을 기록한다.

In [ ]:
# === 원본 모델 범용 응답 기록 ===

print("원본 모델 - 범용 질문 응답:")
print("=" * 60)
baseline_general = []
for prompt in general_prompts:
    response = generate_response(model, tokenizer, prompt)
    baseline_general.append(response)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}")

print("\n" + "=" * 60)
print("원본 모델 - SQL 질문 응답:")
print("=" * 60)
baseline_sql = []
for prompt in sql_prompts:
    response = generate_response(model, tokenizer, prompt)
    baseline_sql.append(response)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}")

---
## 3. SQL 전용 파인튜닝 (망각 유발)

SQL 데이터만으로 파인튜닝하여 의도적으로 Catastrophic Forgetting을 유발한다.  
이것이 **단일 도메인 파인튜닝의 위험성**을 보여주는 실험이다.

In [ ]:
# === SQL 전용 데이터 준비 ===

sql_dataset = load_dataset("b-mc2/sql-create-context", split="train")
sql_data = sql_dataset.shuffle(seed=42).select(range(500))

def format_sql_to_chatml(sample):
    user_msg = f"다음 테이블 구조를 참고하여 SQL 쿼리를 작성하세요.\n\n"
    user_msg += f"테이블: {sample['context']}\n질문: {sample['question']}"
    text = (
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{sample['answer']}<|im_end|>"
    )
    return {"text": text}

sql_formatted = sql_data.map(format_sql_to_chatml)
sql_split = sql_formatted.train_test_split(test_size=0.1, seed=42)

print(f"SQL 학습 데이터: {len(sql_split['train'])}개")
print(f"SQL 검증 데이터: {len(sql_split['test'])}개")
print(f"\n샘플:\n{sql_split['train'][0]['text'][:300]}")

In [ ]:
# === SQL 전용 파인튜닝 (LoRA) ===

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir="./output/sql_only_forgetting",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_available(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    max_length=512,
    dataset_text_field="text",
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=sql_split['train'],
    eval_dataset=sql_split['test'],
    args=training_args,
)

print("SQL 전용 학습 시작...")
result = trainer.train()
print(f"학습 완료! Loss: {result.training_loss:.4f}")

---
## 4. 망각 시연: Before vs After

SQL 파인튜닝 후 범용 질문에 대한 응답 품질이 어떻게 변했는지 확인한다.

In [ ]:
# === 망각 시연 ===

peft_model.eval()

def score_response_quality(response, prompt):
    """간단한 품질 스코어 (0~1)"""
    score = 0.0
    # 최소 길이
    if len(response) > 50:
        score += 0.3
    # SQL 오염 체크 (범용 질문에 SQL 키워드가 나오면 감점)
    sql_keywords = ['SELECT', 'FROM', 'WHERE', 'JOIN', 'INSERT', 'CREATE TABLE']
    if 'sql' not in prompt.lower() and 'query' not in prompt.lower():
        sql_contamination = sum(1 for kw in sql_keywords if kw in response.upper())
        if sql_contamination == 0:
            score += 0.3
    else:
        score += 0.3  # SQL 질문이면 SQL 키워드 있어도 OK
    # 반복 체크
    words = response.split()
    if len(words) > 5 and len(set(words)) / len(words) > 0.3:
        score += 0.2
    # 문장 완성도
    if response.endswith(('.', '!', '?', '```')):
        score += 0.2
    return min(score, 1.0)

print("SQL 파인튜닝 후 - 범용 질문 응답:")
print("=" * 60)
after_general = []
general_scores_before = []
general_scores_after = []

for i, prompt in enumerate(general_prompts):
    response = generate_response(peft_model, tokenizer, prompt)
    after_general.append(response)
    
    score_before = score_response_quality(baseline_general[i], prompt)
    score_after = score_response_quality(response, prompt)
    general_scores_before.append(score_before)
    general_scores_after.append(score_after)
    
    print(f"\nQ: {prompt}")
    print(f"[Before] {baseline_general[i][:150]}")
    print(f"[After]  {response[:150]}")
    print(f"품질: {score_before:.1f} → {score_after:.1f}")

print(f"\n범용 능력 평균: {np.mean(general_scores_before):.2f} → {np.mean(general_scores_after):.2f}")

In [ ]:
# === 시각화: 망각 정도 ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1) 범용 능력 Before/After
x = np.arange(len(general_prompts))
width = 0.35
axes[0].bar(x - width/2, general_scores_before, width, label='Before (원본)', color='#51cf66')
axes[0].bar(x + width/2, general_scores_after, width, label='After (SQL 파인튜닝)', color='#ff6b6b')
axes[0].set_ylabel('품질 점수')
axes[0].set_title('범용 능력 변화 (Catastrophic Forgetting)')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Q{i+1}' for i in range(len(general_prompts))])
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3)

# 2) 능력 분포 변화
categories = ['범용 능력', 'SQL 능력']
before_avg = [np.mean(general_scores_before), 0.5]  # 원본의 SQL은 보통
after_avg = [np.mean(general_scores_after), 0.9]     # SQL 파인튜닝 후

x2 = np.arange(len(categories))
axes[1].bar(x2 - width/2, before_avg, width, label='Before', color='#51cf66')
axes[1].bar(x2 + width/2, after_avg, width, label='After', color='#ff6b6b')
axes[1].set_ylabel('능력 수준')
axes[1].set_title('트레이드오프: SQL ↑ 범용 ↓')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(categories)
axes[1].legend()
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n핵심 관찰:")
print("  - SQL 능력은 확실히 향상")
print("  - 범용 질문에 SQL 패턴이 침투하거나, 답변 품질 저하")
print("  - 이것이 Catastrophic Forgetting의 실체")

---
## 5. 방지 전략과 Replay Buffer

Catastrophic Forgetting을 방지하는 5가지 전략:

| 전략 | 효과 | 난이도 |
|------|------|--------|
| **Low Learning Rate** | 변화폭 제한 | 쉬움 |
| **적은 Epoch** | 과학습 방지 | 쉬움 |
| **Replay Buffer** | **근본 해결** | **중간** |
| **LoRA rank 제한** | 변경 용량 제한 | 쉬움 |
| **정기 범용 평가** | 조기 감지 | 중간 |

**Replay Buffer**: 도메인 데이터에 범용 데이터를 **10~20% 혼합**하여 학습.  
모델이 도메인을 배우면서도 범용 능력을 "잊지 않도록" 상기시킨다.

In [ ]:
# === Replay Buffer 데이터 구성 ===

# 범용 데이터 (Alpaca에서 소량 추출)
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca_sample = alpaca.shuffle(seed=42).select(range(50))  # SQL 450 + 범용 50 = 10% 비율

def format_alpaca_to_chatml(sample):
    if sample.get('input') and sample['input'].strip():
        user_msg = f"{sample['instruction']}\n\nInput: {sample['input']}"
    else:
        user_msg = sample['instruction']
    text = (
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{sample['output']}<|im_end|>"
    )
    return {"text": text}

alpaca_formatted = alpaca_sample.map(format_alpaca_to_chatml)

# Replay Buffer = SQL + 범용 혼합
replay_train = concatenate_datasets([
    sql_split['train'],
    alpaca_formatted
]).shuffle(seed=42)

print(f"Replay Buffer 구성:")
print(f"  SQL 데이터: {len(sql_split['train'])}개 ({len(sql_split['train'])/len(replay_train)*100:.0f}%)")
print(f"  범용 데이터: {len(alpaca_formatted)}개 ({len(alpaca_formatted)/len(replay_train)*100:.0f}%)")
print(f"  총합: {len(replay_train)}개")

In [ ]:
# === Replay Buffer로 재학습 ===
# 새 모델에서 시작 (이전 파인튜닝의 영향 제거)

model2 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model2 = prepare_model_for_kbit_training(model2)
peft_model2 = get_peft_model(model2, lora_config)

training_args2 = SFTConfig(
    output_dir="./output/sql_replay_buffer",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy="no",
    max_length=512,
    dataset_text_field="text",
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)

trainer2 = SFTTrainer(
    model=peft_model2,
    processing_class=tokenizer,
    train_dataset=replay_train,
    args=training_args2,
)

print("Replay Buffer 학습 시작...")
result2 = trainer2.train()
print(f"학습 완료! Loss: {result2.training_loss:.4f}")

In [ ]:
# === Replay Buffer 효과 검증 ===

peft_model2.eval()
replay_scores = []

print("Replay Buffer 모델 - 범용 질문 응답:")
print("=" * 60)

for i, prompt in enumerate(general_prompts):
    response = generate_response(peft_model2, tokenizer, prompt)
    score = score_response_quality(response, prompt)
    replay_scores.append(score)
    print(f"\nQ: {prompt}")
    print(f"[원본]          {baseline_general[i][:120]}")
    print(f"[SQL Only]      {after_general[i][:120]}")
    print(f"[Replay Buffer] {response[:120]}")
    print(f"품질: {general_scores_before[i]:.1f} → {general_scores_after[i]:.1f} → {score:.1f}")

# 3-way 비교 시각화
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(general_prompts))
width = 0.25
ax.bar(x - width, general_scores_before, width, label='원본', color='#51cf66')
ax.bar(x, general_scores_after, width, label='SQL Only', color='#ff6b6b')
ax.bar(x + width, replay_scores, width, label='Replay Buffer', color='#339af0')
ax.set_ylabel('품질 점수')
ax.set_title('범용 능력: 원본 vs SQL Only vs Replay Buffer')
ax.set_xticks(x)
ax.set_xticklabels([f'Q{i+1}' for i in range(len(general_prompts))])
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n범용 능력 평균:")
print(f"  원본:          {np.mean(general_scores_before):.2f}")
print(f"  SQL Only:      {np.mean(general_scores_after):.2f}")
print(f"  Replay Buffer: {np.mean(replay_scores):.2f}")

---
## 6. W&B 실전 연동

`report_to="wandb"` 한 줄로 기본 로깅이 되지만,  
**Custom Callback**으로 더 풍부한 추적이 가능하다:

| 기본 로깅 | Custom Callback |
|-----------|----------------|
| loss, lr | + grad_norm |
| eval_loss | + 생성 샘플 추적 |
| 속도 메트릭 | + 커스텀 메트릭 |

In [ ]:
# === W&B Custom Callback ===
# 실제 실행은 wandb 계정이 필요하므로, 구조를 보여주고 로컬로 시뮬레이션

class WandbCustomCallback(TrainerCallback):
    """W&B에 추가 메트릭을 로깅하는 콜백"""
    
    def __init__(self, model, tokenizer, eval_prompts, log_every_n_steps=50):
        self.model = model
        self.tokenizer = tokenizer
        self.eval_prompts = eval_prompts
        self.log_every_n_steps = log_every_n_steps
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """매 로깅 스텝마다 실행"""
        if logs is None:
            return
        
        # grad_norm 추적 (학습 안정성 지표)
        if 'grad_norm' in logs:
            # wandb.log({"custom/grad_norm": logs['grad_norm']}, step=state.global_step)
            pass  # wandb 미연동 시 패스
    
    def on_epoch_end(self, args, state, control, **kwargs):
        """에폭 끝마다 샘플 생성 로깅"""
        self.model.eval()
        samples = []
        for prompt in self.eval_prompts:
            response = generate_response(self.model, self.tokenizer, prompt, max_new_tokens=100)
            samples.append(f"Q: {prompt}\nA: {response}")
        
        # wandb.log({
        #     "samples/generation": wandb.Table(
        #         columns=["prompt", "response"],
        #         data=[[p, r] for p, r in zip(self.eval_prompts, [s.split('\nA: ')[1] for s in samples])]
        #     )
        # }, step=state.global_step)
        
        print(f"\n[Epoch {state.epoch:.0f}] 샘플 생성:")
        for s in samples[:2]:  # 첫 2개만 출력
            print(f"  {s[:100]}")

print("WandbCustomCallback 정의 완료")
print("\n사용법:")
print('  import wandb')
print('  wandb.init(project="llm-finetune", name="sql-replay-buffer")')
print('  ')
print('  callback = WandbCustomCallback(model, tokenizer, general_prompts)')
print('  trainer = SFTTrainer(..., callbacks=[callback])')
print('  # TrainingArguments에 report_to="wandb" 추가')

In [ ]:
# === W&B 연동 시뮬레이션 (로컬 로그) ===

# 실제 W&B 대시보드에서 보이는 항목을 로컬로 재현
log_history = trainer.state.log_history

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Train Loss
train_steps = [l['step'] for l in log_history if 'loss' in l]
train_loss = [l['loss'] for l in log_history if 'loss' in l]
if train_steps:
    axes[0, 0].plot(train_steps, train_loss, 'b-', alpha=0.7)
    axes[0, 0].set_title('train/loss')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].grid(True, alpha=0.3)

# 2) Learning Rate
lr_steps = [l['step'] for l in log_history if 'learning_rate' in l]
lr_vals = [l['learning_rate'] for l in log_history if 'learning_rate' in l]
if lr_steps:
    axes[0, 1].plot(lr_steps, lr_vals, 'g-')
    axes[0, 1].set_title('train/learning_rate')
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].grid(True, alpha=0.3)

# 3) Eval Loss
eval_steps = [l['step'] for l in log_history if 'eval_loss' in l]
eval_loss = [l['eval_loss'] for l in log_history if 'eval_loss' in l]
if eval_steps:
    axes[1, 0].plot(eval_steps, eval_loss, 'ro-', markersize=8)
    axes[1, 0].set_title('eval/loss')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].grid(True, alpha=0.3)

# 4) Grad Norm (시뮬레이션)
grad_norms = [l.get('grad_norm', None) for l in log_history if 'loss' in l]
grad_norms_clean = [g for g in grad_norms if g is not None]
if grad_norms_clean:
    axes[1, 1].plot(train_steps[:len(grad_norms_clean)], grad_norms_clean, 'm-', alpha=0.7)
    axes[1, 1].set_title('train/grad_norm')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='clip threshold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'grad_norm 미기록\n(W&B 연동 시 자동 추적)', 
                    ha='center', va='center', fontsize=12)
    axes[1, 1].set_title('train/grad_norm')

plt.suptitle('W&B 대시보드 시뮬레이션', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nW&B 대시보드에서 추적하는 핵심 메트릭:")
print("  1. train/loss      - 학습 손실 (감소 추세 확인)")
print("  2. train/lr        - 학습률 스케줄 (코사인 감쇠 확인)")
print("  3. eval/loss       - 검증 손실 (과적합 감지)")
print("  4. train/grad_norm - 그래디언트 크기 (폭발/소실 감지)")

---
## 정리

| 개념 | 핵심 | 대응 |
|------|------|------|
| **Catastrophic Forgetting** | 도메인 학습 시 범용 능력 상실 | Replay Buffer |
| **Replay Buffer** | 도메인 90% + 범용 10% 혼합 | 근본적 해결책 |
| **W&B 기본** | `report_to="wandb"` 한 줄 | loss, lr 자동 추적 |
| **W&B Custom** | TrainerCallback 구현 | grad_norm, 생성 샘플 추적 |

### 핵심 원칙

> **파인튜닝은 "새로운 능력 추가"가 아니라 "능력의 재분배"다.**  
> 도메인 능력을 얻는 대가로 범용 능력을 잃지 않으려면, Replay Buffer로 균형을 잡아라.

### 다음: 02_학습_안정성_디버깅_고급.ipynb

Loss 이상보다 은밀한 3가지 버그를 진단한다:  
padding_side 불일치, BOS/EOS 중복, special token 미등록